In [75]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import date

In [76]:
spark = (SparkSession.builder.
         master("local[*]").
         appName("Task_total_2").
         getOrCreate())

In [77]:
#1.1 Загрузка и вывод схемы
df_csv = spark.read.csv("retail_store_sales.csv", header=True, inferSchema=True)
df_csv.show(5) 
df_csv.printSchema()

+--------------+-----------+-------------+------------+--------------+--------+-----------+--------------+--------+----------------+----------------+
|Transaction ID|Customer ID|     Category|        Item|Price Per Unit|Quantity|Total Spent|Payment Method|Location|Transaction Date|Discount Applied|
+--------------+-----------+-------------+------------+--------------+--------+-----------+--------------+--------+----------------+----------------+
|   TXN_6867343|    CUST_09|   Patisserie| Item_10_PAT|          18.5|    10.0|      185.0|Digital Wallet|  Online|      2024-04-08|            true|
|   TXN_3731986|    CUST_22|Milk Products|Item_17_MILK|          29.0|     9.0|      261.0|Digital Wallet|  Online|      2023-07-23|            true|
|   TXN_9303719|    CUST_02|     Butchers| Item_12_BUT|          21.5|     2.0|       43.0|   Credit Card|  Online|      2022-10-05|           false|
|   TXN_9458126|    CUST_06|    Beverages| Item_16_BEV|          27.5|     9.0|      247.5|   Credit

In [78]:
#1.2 Очистка названий столбцов 
new_columns = [c.strip().replace(" ", "_").lower() for c in df_csv.columns]
df = df_csv.toDF(*new_columns)
print(df.columns)

['transaction_id', 'customer_id', 'category', 'item', 'price_per_unit', 'quantity', 'total_spent', 'payment_method', 'location', 'transaction_date', 'discount_applied']


In [79]:
# 1.3 Преобразование типов данных. Все типы определились корректно т.к. набор данных не большой inferSchema=True отработал корректно, дополнительное преобразование не требуется 
df_csv.printSchema()

root
 |-- Transaction ID: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Item: string (nullable = true)
 |-- Price Per Unit: double (nullable = true)
 |-- Quantity: double (nullable = true)
 |-- Total Spent: double (nullable = true)
 |-- Payment Method: string (nullable = true)
 |-- Location: string (nullable = true)
 |-- Transaction Date: date (nullable = true)
 |-- Discount Applied: boolean (nullable = true)



In [80]:
df.summary().show(truncate=10)

+-------+--------------+-----------+----------+----------+--------------+----------+-----------+--------------+--------+
|summary|transaction_id|customer_id|  category|      item|price_per_unit|  quantity|total_spent|payment_method|location|
+-------+--------------+-----------+----------+----------+--------------+----------+-----------+--------------+--------+
|  count|         12575|      12575|     12575|     11362|         11966|     11971|      11971|         12575|   12575|
|   mean|          NULL|       NULL|      NULL|      NULL|    23.3659...|5.53637...| 129.652...|          NULL|    NULL|
| stddev|          NULL|       NULL|      NULL|      NULL|    10.7435...|2.85788...| 94.7506...|          NULL|    NULL|
|    min|    TXN_100...|    CUST_01| Beverages|Item_10...|           5.0|       1.0|        5.0|          Cash|In-store|
|    25%|          NULL|       NULL|      NULL|      NULL|          14.0|       3.0|       51.0|          NULL|    NULL|
|    50%|          NULL|       N

In [81]:
#2.1 Заполнение отсутствующие Price Per Unit
df = df.withColumn("price_per_unit", F.when(F.col("price_per_unit").isNull()&F.col("quantity").isNotNull()&F.col("total_spent").isNotNull(), F.col("total_spent")/ F.col("quantity")).otherwise(F.col("price_per_unit"))) 
df.summary().show(truncate=10)

+-------+--------------+-----------+----------+----------+--------------+----------+-----------+--------------+--------+
|summary|transaction_id|customer_id|  category|      item|price_per_unit|  quantity|total_spent|payment_method|location|
+-------+--------------+-----------+----------+----------+--------------+----------+-----------+--------------+--------+
|  count|         12575|      12575|     12575|     11362|         12575|     11971|      11971|         12575|   12575|
|   mean|          NULL|       NULL|      NULL|      NULL|    23.3693...|5.53637...| 129.652...|          NULL|    NULL|
| stddev|          NULL|       NULL|      NULL|      NULL|    10.7487...|2.85788...| 94.7506...|          NULL|    NULL|
|    min|    TXN_100...|    CUST_01| Beverages|Item_10...|           5.0|       1.0|        5.0|          Cash|In-store|
|    25%|          NULL|       NULL|      NULL|      NULL|          14.0|       3.0|       51.0|          NULL|    NULL|
|    50%|          NULL|       N

In [32]:
#2.2 Восстановление отсутствующих Item
df_r_item = (df.dropna(subset=["item"]).select("category",F.col("item").alias("new_item"),"price_per_unit")).distinct()
df_join = df.join(df_r_item, on=["category","price_per_unit"], how="left")
df = df_join.withColumn("item", F.coalesce(F.col("item"),F.col("new_item"))).drop("new_item")
df.summary().show(truncate=10)

+-------+----------+--------------+--------------+-----------+----------+----------+-----------+--------------+--------+
|summary|  category|price_per_unit|transaction_id|customer_id|      item|  quantity|total_spent|payment_method|location|
+-------+----------+--------------+--------------+-----------+----------+----------+-----------+--------------+--------+
|  count|     12575|         12575|         12575|      12575|     12575|     11971|      11971|         12575|   12575|
|   mean|      NULL|    23.3693...|          NULL|       NULL|      NULL|5.53637...| 129.652...|          NULL|    NULL|
| stddev|      NULL|    10.7487...|          NULL|       NULL|      NULL|2.85788...| 94.7506...|          NULL|    NULL|
|    min| Beverages|           5.0|    TXN_100...|    CUST_01|Item_10...|       1.0|        5.0|          Cash|In-store|
|    25%|      NULL|          14.0|          NULL|       NULL|      NULL|       3.0|       51.0|          NULL|    NULL|
|    50%|      NULL|          23

In [82]:
#2.3 Заполнение отсутствующих Quantity иTotal Spent
df = df.withColumn("total_spent", F.when(F.col("total_spent").isNull()&F.col("quantity").isNotNull()&F.col("price_per_unit").isNotNull(), F.col("price_per_unit")*F.col("quantity")).otherwise(F.col("total_spent")))
df = df.withColumn("quantity", F.when(F.col("quantity").isNull()&F.col("total_spent").isNotNull()&F.col("price_per_unit").isNotNull(), F.col("total_spent")/F.col("total_spent")).otherwise(F.col("total_spent"))) 
df.summary().show(truncate=10)

+-------+--------------+-----------+----------+----------+--------------+----------+-----------+--------------+--------+
|summary|transaction_id|customer_id|  category|      item|price_per_unit|  quantity|total_spent|payment_method|location|
+-------+--------------+-----------+----------+----------+--------------+----------+-----------+--------------+--------+
|  count|         12575|      12575|     12575|     11362|         12575|     11971|      11971|         12575|   12575|
|   mean|          NULL|       NULL|      NULL|      NULL|    23.3693...|129.652...| 129.652...|          NULL|    NULL|
| stddev|          NULL|       NULL|      NULL|      NULL|    10.7487...|94.7506...| 94.7506...|          NULL|    NULL|
|    min|    TXN_100...|    CUST_01| Beverages|Item_10...|           5.0|       5.0|        5.0|          Cash|In-store|
|    25%|          NULL|       NULL|      NULL|      NULL|          14.0|      51.0|       51.0|          NULL|    NULL|
|    50%|          NULL|       N

In [41]:
df = df.dropna(subset=["category", "Quantity","total_spent","total_spent", "price_per_unit"])
df.summary().show(truncate=10)

+-------+----------+--------------+--------------+-----------+----------+----------+-----------+--------------+--------+
|summary|  category|price_per_unit|transaction_id|customer_id|      item|  quantity|total_spent|payment_method|location|
+-------+----------+--------------+--------------+-----------+----------+----------+-----------+--------------+--------+
|  count|     11971|         11971|         11971|      11971|     11971|     11971|      11971|         11971|   11971|
|   mean|      NULL|    23.3608...|          NULL|       NULL|      NULL|129.652...| 129.652...|          NULL|    NULL|
| stddev|      NULL|    10.7418...|          NULL|       NULL|      NULL|94.7506...| 94.7506...|          NULL|    NULL|
|    min| Beverages|           5.0|    TXN_100...|    CUST_01|Item_10...|       5.0|        5.0|          Cash|In-store|
|    25%|      NULL|          14.0|          NULL|       NULL|      NULL|      51.0|       51.0|          NULL|    NULL|
|    50%|      NULL|          23

In [83]:
#3 Разведочный анализ данных
#3.1. Самые популярные категории товаров:
df_3_1 = df.select("category", "quantity").groupBy("category").agg(F.sum("quantity").alias("total_quantity")).orderBy(F.col("total_quantity").desc())
df_3_1.show(5)

+--------------------+--------------+
|            category|total_quantity|
+--------------------+--------------+
|            Butchers|      208118.0|
|Electric househol...|      203813.5|
|           Beverages|      197047.5|
|           Furniture|      195310.0|
|                Food|      194812.0|
+--------------------+--------------+
only showing top 5 rows



In [84]:
#3.2. Анализ среднего чека: 
#Рассчитайте среднее значение Total Spent для каждого метода оплаты. Округлите до двух знаков после запятой.
df_3_2_1 = df.select("payment_method","total_spent").groupBy("payment_method").agg(F.round(F.avg("total_spent").alias("avg_total_spent"),2))
#Рассчитайте среднее значение Total Spent для каждой места где прошла оплата. Округлите до двух знаков после запятой.
df_3_2_2 = df.select("location","total_spent").groupBy("location").agg(F.round(F.avg("total_spent").alias("avg_location_total_spent"),2))
df_3_2_1.show()
df_3_2_2.show()

+--------------+---------------------------------------------+
|payment_method|round(avg(total_spent) AS avg_total_spent, 2)|
+--------------+---------------------------------------------+
|   Credit Card|                                       129.13|
|Digital Wallet|                                       128.72|
|          Cash|                                       131.05|
+--------------+---------------------------------------------+

+--------+------------------------------------------------------+
|location|round(avg(total_spent) AS avg_location_total_spent, 2)|
+--------+------------------------------------------------------+
|In-store|                                                128.86|
|  Online|                                                130.42|
+--------+------------------------------------------------------+



In [85]:
#4. Генерация признаков
#4.1. Временные признаки: Добавьте два новых столбца на основе Transaction Date
#day_of_week: День недели
#transaction_month: Месяц транзакции 

df = df.withColumn("day_of_week", F.dayofweek(F.col("transaction_date"))).withColumn("transaction_month",F.month(F.col("transaction_date")))
df.show(5)



+--------------+-----------+-------------+------------+--------------+--------+-----------+--------------+--------+----------------+----------------+-----------+-----------------+
|transaction_id|customer_id|     category|        item|price_per_unit|quantity|total_spent|payment_method|location|transaction_date|discount_applied|day_of_week|transaction_month|
+--------------+-----------+-------------+------------+--------------+--------+-----------+--------------+--------+----------------+----------------+-----------+-----------------+
|   TXN_6867343|    CUST_09|   Patisserie| Item_10_PAT|          18.5|   185.0|      185.0|Digital Wallet|  Online|      2024-04-08|            true|          2|                4|
|   TXN_3731986|    CUST_22|Milk Products|Item_17_MILK|          29.0|   261.0|      261.0|Digital Wallet|  Online|      2023-07-23|            true|          1|                7|
|   TXN_9303719|    CUST_02|     Butchers| Item_12_BUT|          21.5|    43.0|       43.0|   Credit

In [99]:
#4.2. Продажи по дням недели
df_4_2 = df.select("day_of_week","total_spent").groupBy("day_of_week").agg(F.avg("total_spent").alias("avg_total_spent_DayWeek")).orderBy("day_of_week")
df_4_2.show()

+-----------+-----------------------+
|day_of_week|avg_total_spent_DayWeek|
+-----------+-----------------------+
|          1|     130.17914746543778|
|          2|     125.56894519740719|
|          3|     129.51061320754718|
|          4|     126.81589713617767|
|          5|       129.275486152033|
|          6|      134.6362058993638|
|          7|     131.49032258064517|
+-----------+-----------------------+



In [93]:
#4.3.Продажи по месяцам
df_4_3 = df.select("transaction_month","total_spent").groupBy("transaction_month").agg(F.avg("total_spent").alias("avg_total_spent_month")).orderBy("transaction_month")
df_4_3.show()

+-----------------+---------------------+
|transaction_month|avg_total_spent_month|
+-----------------+---------------------+
|                1|   134.68803088803088|
|                2|   130.66048034934497|
|                3|   126.83108808290156|
|                4|    131.8137460650577|
|                5|   127.39723926380368|
|                6|   130.94954591321897|
|                7|   126.57266602502406|
|                8|   124.28175403225806|
|                9|    131.4471544715447|
|               10|    127.8517130620985|
|               11|   128.78578947368422|
|               12|   133.15041067761808|
+-----------------+---------------------+



In [100]:
#4.4. Признаки клиента (CLV) 
df_4_4 = df.select("customer_id","total_spent").groupBy("customer_id").agg(F.sum("total_spent").alias("CLV")).orderBy(F.col("CLV").desc())
df_4_4.show(10)

+-----------+-------+
|customer_id|    CLV|
+-----------+-------+
|    CUST_24|68452.0|
|    CUST_08|67351.5|
|    CUST_05|66974.5|
|    CUST_16|65570.5|
|    CUST_13|65037.0|
|    CUST_23|64507.0|
|    CUST_10|63155.5|
|    CUST_15|63117.5|
|    CUST_21|62933.0|
|    CUST_02|62046.5|
+-----------+-------+
only showing top 10 rows

